# Where a coding-algorithm ban takes effect in an LLM

**Demo of artifact `art_8_ULPE8jJQc_`: a KV-cache splice route test (C2) and a compliance-at-R test (C5) for prompt-level algorithm bans in `Qwen/Qwen3-8B`** (bf16, thinking off).

Ask Qwen3-8B for a grid-path solver and it writes breadth-first search (BFS) every time. Add *"Do not use breadth-first search (BFS) for this task."* and the BFS share drops from 100% to 14%. This experiment asks **where inside the model the ban takes effect**. Each ban prompt gets a **neutral twin**: an irrelevant prohibition in the same frame ("Do not use man, the house and the baby for this task."), padded so that the chat-formatted prompt has exactly the ban prompt's token length and identical token ids from `Task:` onward. The KV caches of the two prompts can then be **spliced position by position**.

| arm | cache of the condition span `[p0, p1)` | cache of the task spec + read position R | how generated tokens can learn about the ban |
|---|---|---|---|
| **a** full ban | ban | ban | every route |
| **b** neutral twin (baseline) | neutral | neutral | no route |
| **c** PROMPT ROUTE | neutral | ban | only through prompt states that already read the ban |
| **d** DIRECT ROUTE | ban | neutral | only by attending to the ban tokens themselves |
| e / f | neutral + ban R only / neutral + ban task-spec states only | | splits up the prompt route |
| K0 / K1 / K2, B1-B3 | ban cache with the condition span hidden from generated tokens (all / first 16 / after 16; layer bands) | | attention knockouts |

**Full-run results** (two tasks: T1 grid path with "ban BFS", T2 weighted shortest path with "ban Dijkstra"; 5 frozen templates):
- **Primary readout:** the FRLP drop of the banned algorithm over 30 self-written programs. It is carried by the **prompt route**: pooled `F_prompt = 0.85 [0.61, 1.20]`, `F_direct = 0.00 [-0.08, 0.12]`.
- **Behaviour (T1):** N = 100 per arm qualifies the T1 result. The BFS share under arms a/b/c/d is 0.14/1.00/0.57/0.70.
- **Mechanism:** the ban vetoes the default first line `from collections import deque`, and that veto is carried entirely by the prompt states.

**This notebook follows the stages of the artifact's `method.py`, using its original code:**
1. **S1 prompts.** Rebuild the frozen ban prompt and its two neutral twins from the demo data with the Qwen3 tokenizer. Check the length-matching rules and that the token ids hash to the frozen values.
2. **S3 engine, tests, S4, S5 (live, toy scale).** Load a small stand-in of the same model family (`Qwen/Qwen3-0.6B`, identical tokenizer). Run the original engine: prompt prefill, position-wise KV splicing, condition-span attention knockouts, teacher-forced scoring and common-random-numbers sampling. This covers the U3 engine checks, CRN behaviour sampling and the natural-proposal FRLP readout.
3. **S8 analysis (real data).** Run the pre-registered analysis and the supplementary statistics on a curated subset of the real Qwen3-8B behaviour samples: 98 samples = 14 matched CRN streams × 7 arms, task T1. Then run the synthetic recovery checks of the FRLP estimator and of the C5 test, and apply the frozen decision rules.
4. **S9 figures.** Plot the demo results next to the full-run results.

> The live numbers in step 2 come from a 0.6B stand-in model scoring a few programs and sampling a few tokens. They show that the machinery works; they are **not** the paper's result. For a full-scale run, set the config cell back to the original values (`MODEL_ID = "Qwen/Qwen3-8B"`, 1024 new tokens, 20 streams per template, all five templates) on a 24 GB GPU.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install
_pip('loguru==0.7.3')

# numpy, pandas, scipy, matplotlib, torch, transformers, tokenizers, huggingface_hub — pre-installed on Colab,
# install locally only (Colab's exact versions; torch is the CPU wheel Colab lists)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0')
    _pip('--index-url', 'https://download.pytorch.org/whl/cpu', 'torch==2.9.0+cpu')
    _pip('transformers==5.0.0', 'tokenizers==0.22.2', 'huggingface_hub==1.4.0')

In [ ]:
from __future__ import annotations

import contextlib
import gc
import hashlib
import json
import math
import os
import re
import sys
import time
from collections import Counter, defaultdict
from typing import Any, Iterator

import numpy as np
import torch
from loguru import logger
from scipy import stats

# notebook: tables, plots and quiet model loading
import matplotlib.pyplot as plt
import pandas as pd
import transformers

transformers.utils.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
# cfg.setup_logging without the rotating log file: loguru to stdout
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| numpy", np.__version__)

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-295f8e-prompt-bans-substitute-not-just-subtract/fork/run_jSassd-YWLVC/round-2/experiment-8/demo/mini_demo_data.json"
import http.client
import json
import urllib.request
from pathlib import Path

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=15) as response:
            return json.loads(response.read().decode())
    # Network errors, timeouts, a placeholder URL, bad JSON, a cut-off response.
    except (OSError, ValueError, http.client.HTTPException) as e:
        print("GitHub data unavailable, using local mini_demo_data.json:", e)
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
REF = data["metadata"]["full_run_reference"]   # aggregated full-run numbers (Qwen3-8B, both tasks)
EXAMPLES = data["datasets"][0]["examples"]     # real Qwen3-8B behaviour samples, task T1
print(data["datasets"][0]["dataset"], "-", len(EXAMPLES), "examples")
print("selection:", data["metadata"]["selection"]["rule"])
print("units (template, CRN stream):", data["metadata"]["selection"]["units"])
print("full-run model:", REF["model"], "@", REF["model_revision"][:10], "|", REF["dtype"], "| thinking", REF["thinking"])

## Configuration

All tunable parameters are in this cell, and each comment gives the original value. The notebook was first tested at the smallest values that still produce every output: 8 new tokens, 2 streams, 1 test program, 1 program per noun, 200 bootstrap replicates (about 3 min on 2 CPU cores). It was then scaled in two rounds to the values below (about 5-6 min on 2 CPU cores; much faster on a GPU). The live part uses `Qwen/Qwen3-0.6B`, which shares Qwen3-8B's tokenizer and therefore the frozen prompts' token ids. It runs on a CPU in seconds per arm. The real samples analysed in stage S8 come from the full Qwen3-8B run, whatever this cell says.

In [ ]:
# ---------------------------------------------------------------- model for the live part (S3 engine, tests, S4, S5)
MODEL_ID = "Qwen/Qwen3-0.6B"   # original: "Qwen/Qwen3-8B" (bf16, ~16.4 GB, needs a 24 GB GPU); same tokenizer
MODEL_REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"   # original: "b968826d9c46dd6066d109eabc6255188de91218"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32   # original: torch.bfloat16 on the GPU
MAX_NEW_TOKENS = 32            # original: 1024 (the opening-line fork happens within the first ~10 tokens)
TEMPERATURE = 1.0              # original: 1.0 (T = 1, top-p 1, no top-k, thinking off)
# ---------------------------------------------------------------- demo scale
TASKS = ("t1",)                # original: ("t1", "t2"); the demo data holds task T1 (ban BFS) only
DEMO_TEMPLATES = ["W1"]        # original: all five templates W1-W5
BEH_PER_TEMPLATE = 6           # original: 20 CRN streams per (task, template); first half noun A, second half noun B
N_TEST_PROGRAMS = 3            # original U3 test: 3 programs (2 for the splice checks)
N_NAT_PROGRAMS_PER_NOUN = 10   # original: every arm-b BFS sample (~10 per noun and template; the demo data has 1-2)
N_LAYERS = 28                  # Qwen3-0.6B; original Qwen3-8B: 36 (re-read from the loaded model below)
# ---------------------------------------------------------------- statistics
N_BOOT = 2000                  # original: 2000 bootstrap replicates
N_PERM_C5 = 10000              # original: 10000 stratified permutations (c5_test default)
RNG_SEED = 20261020            # original analysis seed
print(f"device={DEVICE} dtype={DTYPE} model={MODEL_ID}")

## Frozen protocol (`cfg.py`)

The plan froze these: the display names of the algorithms, the five ban templates W1-W5 (each a ban / mention / neutral triple), the forced paraphrases, the signature vocabularies and the decision thresholds. They are copied unchanged from `cfg.py`, together with the small utilities the stages use. `ALGOS` lists the mechanism-level labels the canonical labeller can assign.

In [ ]:
IM_END = 151645
ENDOFTEXT = 151643
DECODING = "T1.0_topp1.0_topk-off_max1024_thinkoff"

X_KEY = {"t1": "bfs", "t2": "dijkstra"}
SUB_KEY = {"t1": "dfs", "t2": "bf_spfa"}  # the dominant substitute in iteration 1
DISPLAY: dict[str, dict[str, str]] = {
    "t1": {
        "bfs": "breadth-first search (BFS)",
        "dfs": "depth-first search (DFS)",
        "bibfs": "bidirectional BFS",
        "iddfs": "iterative deepening DFS (IDDFS)",
        "astar_gbfs": "A* search",
        "dijkstra": "unit-weight Dijkstra",
    },
    "t2": {
        "dijkstra": "Dijkstra's algorithm",
        "bf_spfa": "Bellman-Ford or SPFA",
        "bidijkstra": "bidirectional Dijkstra",
        "floyd": "Floyd-Warshall",
        "dial": "Dial's bucket-queue algorithm",
    },
}
ALGOS = {t: list(DISPLAY[t].keys()) for t in TASKS}
SURVIVORS = {t: [a for a in ALGOS[t] if a != X_KEY[t]] for t in TASKS}

# W1-W5: (ban, mention, neutral) verbatim from the shared protocol v2.
TEMPLATES: dict[str, tuple[str, str, str]] = {
    "W1": ("Do not use {X} for this task.", "You could use {X} for this task.", "Do not use {N} for this task."),
    "W2": ("Your solution must not rely on {X}.", "Your solution may rely on {X}.", "Your solution must not rely on {N}."),
    "W3": ("Avoid {X} entirely.", "Consider {X} first.", "Avoid {N} entirely."),
    "W4": ("{X} is not allowed here.", "{X} is allowed here.", "{N} is not allowed here."),
    "W5": ("Write a correct solution that does not use {X}.", "Write a correct solution that may use {X}.",
           "Write a correct solution that does not use {N}."),
}
TEMPLATE_IDS = list(TEMPLATES.keys())

FORCED_PARAPHRASES = [
    "Implement {j} for this task.",
    "Use {j} to solve this task.",
    "Solve this task with {j}.",
    "Write an implementation of {j} for this task.",
    "Your solution should use {j}.",
    "Apply {j} to this task.",
    "Base your solution on {j}.",
    "Solve the task below using {j}.",
    "Implement the task with {j}.",
    "Write the solution using {j}.",
]

# Signature vocabularies (frozen). Used only for per-token localisation and the lens.
SIGNATURE_VOCAB: dict[str, list[str]] = {
    "bfs": ["deque", "popleft", "collections"],
    "dfs": ["stack", "pop()", "setrecursionlimit"],
    "dijkstra": ["heapq", "heappush", "heappop", "heapify"],
    "astar_gbfs": ["heapq", "heuristic", "abs("],
    "bf_spfa": ["deque", "popleft", "in_queue", "inq", "range(n - 1)"],
}

MAIN_ARMS = ("a", "b", "c", "d")
X_ONLY_ARMS = ("e", "f", "pl", "K0", "K1", "K2", "B1", "B2", "B3", "K0b")
KO_WINDOW = 16
# notebook: thirds of the model's depth; for N_LAYERS = 36 this is the original {0-11, 12-23, 24-35}
LAYER_BANDS = {"B1": list(range(0, N_LAYERS // 3)), "B2": list(range(N_LAYERS // 3, 2 * N_LAYERS // 3)),
               "B3": list(range(2 * N_LAYERS // 3, N_LAYERS))}
N_PROGRAMS = 30

THRESHOLDS = {
    "F_confirm": 2.0 / 3.0,
    "F_ci_lower": 0.5,
    "placebo_max_frac": 0.20,
    "ess_flag": 5.0,
    "c5_rho": -0.5,
    "c5_alpha": 0.05,
    "gate_avoidance": 0.5,
    "gate_min_nonbanned": 20,
    "crn_floor_min": 0.95,
    "veto_log_rank": 1.0,
}
TEMPLATE_TAIL = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"


def model_path() -> str:
    """BRT_MODEL_PATH (a local copy of the pinned snapshot) if set; else the Hub id (loaded at MODEL_REVISION)."""
    return os.environ.get("BRT_MODEL_PATH") or MODEL_ID   # notebook: no HF-cache snapshot lookup


def hf_kwargs() -> dict[str, str]:
    """Pin the Hub revision whenever the model is loaded by id."""
    return {"revision": MODEL_REVISION} if MODEL_REVISION and model_path() == MODEL_ID else {}


def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()


def stable_seed(*parts: Any) -> int:
    """Deterministic 63-bit seed from arbitrary parts (sha256 of their '|'-joined repr)."""
    return int(hashlib.sha256("|".join(str(p) for p in parts).encode()).hexdigest(), 16) % (2 ** 63)


def wilson_ci(k: int, n: int, z: float = 1.959963984540054) -> tuple[float, float]:
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, c - h), min(1.0, c + h))


def cap_first(s: str) -> str:
    return s[:1].upper() + s[1:] if s else s


def chat_ids(tok: Any, user_text: str) -> list[int]:
    ids = tok.apply_chat_template([{"role": "user", "content": user_text}], tokenize=True,
                                  add_generation_prompt=True, enable_thinking=False)
    if isinstance(ids, dict) or hasattr(ids, "keys"):   # notebook: transformers 5 returns a BatchEncoding
        ids = ids["input_ids"]
    return list(ids)

## Stage map of `method.py`

`method.py` is the artifact's entry point. Each stage is a subcommand, so the GPU stages run as separate processes. This notebook runs the stages in the same order, in cells:

| `method.py` stage | what it does in the full run | here |
|---|---|---|
| `s0` | snapshot of the iteration-1 inputs, suite copy, labeller re-check (0.965 vs gold) | not run (inputs come from other artifacts) |
| `s1` | frozen prompts, length-matched neutral twins, forced prompts | **run** for the demo templates, from the data |
| `s2` | 323 forced programs (30 per algorithm), labelling, name scrub (GPU) | not run; arm-b samples from the data serve as programs |
| `prereg` | sha256 commit of the decision rules, prompts and `analysis.py` before any arm is scored | hash shown below |
| `tests` | U0-U9 engine / sampler / knockout / analysis tests | **U3 run live**, U9 run in S8 |
| `s4` / `s4label` | CRN behaviour sampling, 100 samples per arm and task; canonical labeller | **run live at toy scale** (opening line only) |
| `s5` | FRLP scoring of every arm (GPU) | **natural-proposal FRLP run live at toy scale** |
| `s6` / `s7` | logit and tuned lens at the commitment token; C5 compliance at R | full-run results in the summary |
| `s8` | every statistic and the frozen decision (`analysis.py`) | **run** on the real Qwen3-8B samples |
| `s9` | `method_out.json` and figures | **run** (figures inline) |

The helper `_time` is `method.py`'s stage timer. It records each stage's duration; here the durations are kept in memory instead of `results/stage_times.jsonl`.

In [ ]:
STAGE_TIMES: list[dict[str, Any]] = []   # notebook: method.py appends these rows to results/stage_times.jsonl


def _time(stage: str, fn) -> Any:
    t0 = time.time()
    logger.info(f"=== stage {stage} start")
    out = fn()
    dt = time.time() - t0
    STAGE_TIMES.append({"stage": stage, "seconds": round(dt, 1)})
    logger.info(f"=== stage {stage} done in {dt / 60:.1f} min")
    return out   # notebook: return the stage result so later cells can use it


print("pre-registration sha256 (decision rules + frozen prompts + analysis.py):", REF["prereg_sha256"])

## Demo data → the `data/samples.jsonl` schema

Each example in the demo data is one real Qwen3-8B behaviour sample, as exported in `method_out.json`:
- `output`: the mechanism-level label from the canonical labeller (runtime tracer + AST, with a blind judge only when they disagree);
- `metadata_code`: the generated program;
- `metadata_arm`, `metadata_template`, `metadata_sample_id`: the arm, the template and the CRN stream.

All arms of one stream share one uniform random number per decoding step, so samples with the same `(template, sample_id)` form **matched pairs** across arms. The adapter below maps the examples back to the row schema that `analysis.py` and `supplementary.py` read.

In [ ]:
def samples_from_examples(examples: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """notebook: method_out examples -> rows of data/samples.jsonl (the schema analysis.py / supplementary.py read)."""
    rows = []
    for e in examples:
        noun = None if e["predict_noun"] == "-" else e["predict_noun"]
        rows.append({"task": e["metadata_task"], "condition": e["metadata_arm"], "template": e["metadata_template"],
                     "sample_id": e["metadata_sample_id"], "seed": e["metadata_seed"],
                     "decoding": e["metadata_decoding"], "label": e["output"],
                     "labeller_source": e["metadata_labeller_source"], "passed": e["predict_passed"] == "True",
                     "banned": e["predict_banned"] == "True", "noun": noun,
                     # original rule: streams s < 10 use neutral noun A, s >= 10 noun B (arms b, c, d only)
                     "noun_slot": "-" if noun is None else ("A" if e["metadata_sample_id"] < 10 else "B"),
                     "code": e["metadata_code"], "code_sha256": e["metadata_code_sha256"],
                     "n_tokens": e["metadata_n_tokens"], "finish_reason": e["metadata_finish_reason"]})
    return rows


SAMPLES = samples_from_examples(EXAMPLES)
ARM_ORDER = ["b", "d", "c", "a", "K0", "K1", "K2"]
tab = pd.crosstab(pd.Series([s["condition"] for s in SAMPLES], name="arm"),
                  pd.Series([s["label"] for s in SAMPLES], name="label"))
print("Mechanism labels per arm (14 matched CRN streams per arm):")
print(tab.loc[ARM_ORDER].to_string())
print("\nFirst lines of one matched stream (W1, s=6) under every arm:")
for arm in ARM_ORDER:
    s = next(x for x in SAMPLES if x["condition"] == arm and x["template"] == "W1" and x["sample_id"] == 6)
    print(f"  {arm:>3} [{s['label']:>4}] {s['code'].strip().splitlines()[0]}")

## Stage S1: frozen prompts and length-matched neutral twins (`build_prompts.py`)

A prompt is `[condition segment] + blank line + [task SPEC]`, wrapped in Qwen3's chat template with thinking switched off. `_positions` finds the condition span:
- `p0` is the end of the chat header;
- `p1` is the first token of the `Task:` paragraph.

A neutral twin is accepted only if all of these hold:
- it has the ban prompt's length `P`, `p0` and `p1`;
- its token ids are identical from `p1` onward;
- it differs only inside `[p0, p1)`.

These are the rules of `build_prompts._match_neutral`. The demo reads the ban segment and the two neutral segments (nouns A = "man", B = "guy", padded with companion nouns) from the examples. It re-tokenizes them and checks every rule. It also checks that the sha256 of the token ids matches the frozen prompts the experiment used.

In [ ]:
def parse_input(inp: str) -> dict[str, str]:
    """notebook: example input = '[arm] [template W] [ban segment: ...] [neutral segment: ...]' + blank line + prompt."""
    m = re.match(r"\[(.*?)\] \[template (W\d)\] \[ban segment: (.*?)\] \[neutral segment: (.*?)\]\n\n(.*)", inp, re.S)
    return {"arm_desc": m.group(1), "template": m.group(2), "ban_segment": m.group(3),
            "neutral_segment": m.group(4), "user_text": m.group(5)}


# notebook: the canonical exp1 SPEC is the ban prompt without its first paragraph
SPEC = {"t1": parse_input(EXAMPLES[0]["input"])["user_text"].split("\n\n", 1)[1]}


def spec_text(task: str) -> str:
    """Canonical exp1 SPEC (DESCRIPTION + blank line + FORMAT); original: read from the snapshotted conditions.json."""
    return SPEC[task]


def user_prompt(segment: str, task: str) -> str:
    """exp1 rule: [condition segment, first character upper-cased] + blank line + [SPEC]."""
    return cap_first(segment) + "\n\n" + spec_text(task)


def load_tokenizer():
    from transformers import AutoTokenizer

    return AutoTokenizer.from_pretrained(model_path(), **hf_kwargs())


def _positions(tok, ids: list[int], task: str) -> tuple[int, int]:
    head = tok("<|im_start|>user\n", add_special_tokens=False)["input_ids"]
    assert ids[:len(head)] == head, "unexpected chat header"
    suf = tok(spec_text(task) + TEMPLATE_TAIL, add_special_tokens=False)["input_ids"]
    p1 = len(ids) - len(suf)
    assert ids[p1:] == suf, "SPEC + template tail do not tokenize identically in context"
    return len(head), p1


def _prompt_record(tok, segment: str, task: str) -> dict[str, Any]:
    text = user_prompt(segment, task)
    ids = chat_ids(tok, text)
    p0, p1 = _positions(tok, ids, task)
    return {"segment": cap_first(segment), "user_text": text, "ids": ids, "P": len(ids), "p0": p0, "p1": p1,
            "ids_sha256": sha256_text(json.dumps(ids))}


def stage_prompts_demo(tok) -> dict[str, Any]:
    """notebook: S1 for DEMO_TEMPLATES. Ban prompt + the two neutral twins read from the arm-b examples, accepted with
    the rules of build_prompts._match_neutral and compared with the frozen prompts' id hashes."""
    fp: dict[str, Any] = {"tasks": {}}
    for task in TASKS:
        tdata: dict[str, Any] = {"templates": {}, "nouns": {}}
        for w in DEMO_TEMPLATES:
            exs = [e for e in EXAMPLES if e["metadata_task"] == task and e["metadata_template"] == w]
            ban = _prompt_record(tok, parse_input(exs[0]["input"])["ban_segment"], task)
            tw = {"ban": ban}
            for slot in ("A", "B"):
                e = next(e for e in exs if e["metadata_arm"] == "b" and (e["metadata_sample_id"] < 10) == (slot == "A"))
                rec = _prompt_record(tok, parse_input(e["input"])["neutral_segment"], task)
                # build_prompts._match_neutral acceptance rules
                assert rec["P"] == ban["P"] and rec["p1"] == ban["p1"] and rec["p0"] == ban["p0"]
                assert rec["ids"][rec["p1"]:] == ban["ids"][ban["p1"]:]
                diff = [i for i, (a, b) in enumerate(zip(rec["ids"], ban["ids"])) if a != b]
                assert diff and min(diff) >= ban["p0"] and max(diff) < ban["p1"]
                rec.update({"noun": e["predict_noun"], "diff_span": [min(diff), max(diff) + 1],
                            "n_diff_positions": len(diff)})
                tw[f"neutral_{slot}"] = rec
                tdata["nouns"][slot] = {"noun": e["predict_noun"]}
            frozen = REF["frozen_prompts_t1"][w]
            for k in ("ban", "neutral_A", "neutral_B"):
                tw[k]["matches_frozen_ids"] = tw[k]["ids_sha256"] == frozen[k]["ids_sha256"]
            tdata["templates"][w] = tw
            b = tw["ban"]
            logger.info(f"[{task} {w}] P={b['P']} p0={b['p0']} p1={b['p1']}  ban: {b['segment']}  |  "
                        f"neutral A: {tw['neutral_A']['segment']} diff={tw['neutral_A']['diff_span']}  |  "
                        f"neutral B: {tw['neutral_B']['segment']}")
        fp["tasks"][task] = tdata
    return fp


tok = load_tokenizer()
fp = _time("s1", lambda: stage_prompts_demo(tok))
tw = fp["tasks"]["t1"]["templates"][DEMO_TEMPLATES[0]]
print("token ids identical to the frozen prompts:",
      {k: tw[k]["matches_frozen_ids"] for k in ("ban", "neutral_A", "neutral_B")})
p0, p1 = tw["ban"]["p0"], tw["ban"]["p1"]
span = pd.DataFrame({k: [tok.decode([i]) for i in tw[k]["ids"][p0:p1]] for k in ("ban", "neutral_A", "neutral_B")},
                    index=range(p0, p1))
span.index.name = "position"
print(f"\nCondition span [p0, p1) = [{p0}, {p1}) token by token (everything else is identical):")
print(span.to_string())

## Stage S3: the engine (`engine.py`), part 1: knockout attention

The knockout is a custom attention function, registered with transformers under the name `ko_sdpa`. While a knockout is active, attention is causal **and** *not* (key column in the condition span `[p0, p1)` **and** query row in the generated-token window `[g_lo, g_hi)` **and** layer in the chosen set). Otherwise stock SDPA runs unchanged.
- K0 hides the ban from every generated token.
- K1 hides it from the first 16 generated tokens; K2 from the rest.
- B1/B2/B3 restrict K0 to one band of layers.

In [ ]:
KO: dict[str, Any] = {"active": False, "p0": 0, "p1": 0, "P": 0, "g_lo": 0, "g_hi": 10 ** 9, "layers": None}
_REGISTERED = False


def _ko_on(module: Any) -> bool:
    return bool(KO["active"]) and (KO["layers"] is None or module.layer_idx in KO["layers"])


def _ko_allowed(q_len: int, k_len: int, device: torch.device) -> torch.Tensor:
    offset = k_len - q_len
    qpos = torch.arange(offset, k_len, device=device)
    kpos = torch.arange(k_len, device=device)
    allowed = kpos[None, :] <= qpos[:, None]
    g = qpos - int(KO["P"])
    qblock = (g >= int(KO["g_lo"])) & (g < int(KO["g_hi"]))
    kblock = (kpos >= int(KO["p0"])) & (kpos < int(KO["p1"]))
    return allowed & ~(qblock[:, None] & kblock[None, :])


def _combine(allowed: torch.Tensor, attention_mask: torch.Tensor | None, k_len: int) -> torch.Tensor:
    allowed = allowed[None, None]
    if attention_mask is not None:
        am = attention_mask[:, :, :, :k_len]
        if am.dtype == torch.bool:
            allowed = allowed & am
        else:
            allowed = allowed & (am == 0)
    return allowed


def ko_sdpa(module, query, key, value, attention_mask, **kwargs):
    from transformers.integrations.sdpa_attention import sdpa_attention_forward

    if _ko_on(module):
        q_len, k_len = query.shape[2], key.shape[2]
        allowed = _combine(_ko_allowed(q_len, k_len, query.device), attention_mask, k_len)
        kwargs.pop("is_causal", None)
        return sdpa_attention_forward(module, query, key, value, allowed, is_causal=False, **kwargs)
    return sdpa_attention_forward(module, query, key, value, attention_mask, **kwargs)


def register() -> None:
    global _REGISTERED
    if _REGISTERED:
        return
    from transformers import AttentionInterface
    from transformers.masking_utils import AttentionMaskInterface, sdpa_mask

    AttentionInterface.register("ko_sdpa", ko_sdpa)
    AttentionMaskInterface.register("ko_sdpa", sdpa_mask)   # notebook: the eager twin (U4 test only) is not registered
    _REGISTERED = True


@contextlib.contextmanager
def knockout(cfg: dict[str, Any] | None) -> Iterator[None]:
    """cfg: {'p0','p1','P','g_lo','g_hi','layers'} or None (no knockout)."""
    old = dict(KO)
    try:
        if cfg is not None:
            KO.update({"active": True, "g_lo": 0, "g_hi": 10 ** 9, "layers": None})
            KO.update(cfg)
        else:
            KO["active"] = False
        yield
    finally:
        KO.clear()
        KO.update(old)


def set_attn(model: Any, name: str) -> None:
    model.config._attn_implementation = name
    for m in model.modules():
        if hasattr(m, "config") and m.config is not model.config and hasattr(m.config, "_attn_implementation"):
            m.config._attn_implementation = name

### Engine, part 2: model loading, prefill and KV splicing

`prefill` runs one prompt and keeps three things: every layer's key/value cache `[1, kv_heads, P, head_dim]`, the logits at the read position R (the last prompt token), and `P`. `splice(A, B, cut)` takes positions `[0, cut)` from cache A and `[cut, P)` from cache B in **every** layer. Both prompts have identical token ids from `p1` onward, so a splice at `p1` gives a well-formed cache. In it, the condition span comes from one prompt, while the task-spec states that *read* that span come from the other.

In [ ]:
def load_model(vram_fraction: float = 0.95):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    register()
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_path(), **hf_kwargs())
    # notebook: dtype/device from the config cell; original: dtype=torch.bfloat16, device_map={"": 0}
    model = AutoModelForCausalLM.from_pretrained(model_path(), dtype=DTYPE, attn_implementation="ko_sdpa",
                                                 **hf_kwargs()).to(DEVICE)
    model.eval()
    if DEVICE == "cuda":   # notebook: the VRAM cap only applies on a GPU
        torch.cuda.set_per_process_memory_fraction(vram_fraction, 0)
        free, total = torch.cuda.mem_get_info(0)
        logger.info(f"free VRAM {free / 1e9:.1f}/{total / 1e9:.1f} GB")
    logger.info(f"HF model loaded in {time.time() - t0:.0f}s; attn={model.config._attn_implementation}")
    return tok, model


def _kv_list(cache: Any) -> list[tuple[torch.Tensor, torch.Tensor]]:
    if hasattr(cache, "layers"):
        return [(L.keys, L.values) for L in cache.layers]
    return list(zip(cache.key_cache, cache.value_cache))


def build_cache(kv: list[tuple[torch.Tensor, torch.Tensor]], batch: int = 1):
    from transformers.cache_utils import DynamicCache

    cache = DynamicCache()
    for l, (K, V) in enumerate(kv):
        if K.shape[0] != batch:
            K = K.expand(batch, -1, -1, -1)
            V = V.expand(batch, -1, -1, -1)
        cache.update(K.contiguous(), V.contiguous(), l)
    return cache


@torch.no_grad()
def prefill(model: Any, ids: list[int], hidden_layers: list[int] | None = None) -> dict[str, Any]:
    """Returns {'kv': per-layer (K, V) [1, 8, P, 128] on GPU, 'r_logits': float32 [V], 'P', 'hR': {l: float32}}."""
    from transformers.cache_utils import DynamicCache

    inp = torch.tensor([ids], device=model.device)
    cache = DynamicCache()
    out = model.model(input_ids=inp, past_key_values=cache, use_cache=True,
                      output_hidden_states=hidden_layers is not None)
    r_logits = model.lm_head(out.last_hidden_state[:, -1]).float()[0]
    kv = [(K.clone(), V.clone()) for K, V in _kv_list(cache)]
    res = {"kv": kv, "r_logits": r_logits, "P": len(ids)}
    if hidden_layers is not None:
        res["hR"] = {l: out.hidden_states[l][0, -1].float().cpu().numpy() for l in hidden_layers}
    del cache, out
    return res


def splice(kv_a: list, kv_b: list, cut: int) -> list:
    """Positions [0, cut) from A and [cut, P) from B, every layer."""
    return [(torch.cat([Ka[:, :, :cut], Kb[:, :, cut:]], dim=2), torch.cat([Va[:, :, :cut], Vb[:, :, cut:]], dim=2))
            for (Ka, Va), (Kb, Vb) in zip(kv_a, kv_b)]


def replace_positions(kv_base: list, kv_src: list, positions: list[int]) -> list:
    out = []
    idx = torch.tensor(positions, device=kv_base[0][0].device)
    for (Kb, Vb), (Ks, Vs) in zip(kv_base, kv_src):
        K, V = Kb.clone(), Vb.clone()
        K[:, :, idx] = Ks[:, :, idx]
        V[:, :, idx] = Vs[:, :, idx]
        out.append((K, V))
    return out

### Engine, part 3: teacher-forced scoring and common-random-numbers sampling

- **`score`** places a token sequence after an arm's cache and returns per-token log-probabilities. Token 0 is scored with the arm's R logits. Scoring runs at batch 1, so a program's score depends only on the cache content.
- **`sample`** draws tokens by inverse-CDF sampling in vocabulary order from **one shared uniform per (stream, step)**. Every arm of a stream sees the same random numbers, so two arms can only produce different text because their caches differ. With `prune=False`, finished rows stay in the batch, keeping the batch shape identical across arms.
- **`logmeanexp`** and **`ess`** are the importance-sampling helpers.

In [ ]:
@torch.no_grad()
def _token_logprobs(model: Any, h: torch.Tensor, tgt: torch.Tensor, chunk: int = 256) -> torch.Tensor:
    out = []
    for i in range(0, h.shape[0], chunk):
        logits = model.lm_head(h[i:i + chunk]).float()
        out.append(torch.log_softmax(logits, dim=-1).gather(-1, tgt[i:i + chunk, None])[:, 0])
    return torch.cat(out) if out else torch.zeros(0, device=h.device)


@torch.no_grad()
def score(model: Any, arm: dict[str, Any], seqs: list[list[int]], ko: dict[str, Any] | None = None,
          token_budget: int = 9000, max_batch: int = 1) -> list[np.ndarray]:
    """Teacher-forced per-token log-probs of each token sequence placed after the arm's prompt cache.
    Token 0 is scored with the arm's R logits; token t >= 1 with the hidden state at position P + t - 1.
    Default batch 1: a program's score then depends only on the cache content (identical GEMM/attention shapes in
    every arm of a template), never on which other programs share its batch (bf16 kernel choice is shape-dependent)."""
    kv, r_logits = arm["kv"], arm["r_logits"]
    P = kv[0][0].shape[2]
    r_lp = torch.log_softmax(r_logits.float(), dim=-1)
    order = sorted(range(len(seqs)), key=lambda i: -len(seqs[i]))
    res: list[np.ndarray | None] = [None] * len(seqs)
    i = 0
    bs = max_batch
    while i < len(order):
        T = len(seqs[order[i]])
        b = max(1, min(bs, token_budget // max(T, 1), len(order) - i))
        idxs = order[i:i + b]
        try:
            inp = torch.full((b, T), IM_END, dtype=torch.long, device=model.device)
            for r, k in enumerate(idxs):
                inp[r, :len(seqs[k])] = torch.tensor(seqs[k], device=model.device)
            cache = build_cache(kv, b)
            pos = torch.arange(P, P + T, device=model.device)[None].expand(b, -1)
            kcfg = None if ko is None else {**ko, "P": P}
            with knockout(kcfg):
                h = model.model(input_ids=inp, past_key_values=cache, position_ids=pos, use_cache=True).last_hidden_state
            for r, k in enumerate(idxs):
                n = len(seqs[k])
                lp = _token_logprobs(model, h[r, :n - 1], inp[r, 1:n])
                first = r_lp[seqs[k][0]].reshape(1)
                res[k] = torch.cat([first, lp]).float().cpu().numpy()
            del cache, h, inp
            i += b
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            gc.collect()
            if bs == 1 and token_budget < 600:
                raise
            bs = max(1, b // 2)
            token_budget = max(512, token_budget // 2)
            logger.warning(f"OOM in score: batch -> {bs}, token budget -> {token_budget}")
    return res  # type: ignore[return-value]


def u_stream(*key: Any, n: int = MAX_NEW_TOKENS) -> np.ndarray:
    return np.random.Generator(np.random.PCG64(stable_seed(*key))).random(n)


def _inverse_cdf(probs: torch.Tensor, u: torch.Tensor) -> torch.Tensor:
    cdf = probs.cumsum(dim=-1)
    cdf = cdf / cdf[:, -1:]
    tok = torch.searchsorted(cdf, u[:, None].to(cdf.dtype).contiguous()).squeeze(-1)   # notebook: .contiguous()
    return tok.clamp_(max=probs.shape[-1] - 1)


@torch.no_grad()
def sample(model: Any, rows: list[dict[str, Any]], max_new: int = MAX_NEW_TOKENS, temperature: float = TEMPERATURE,
           ko: dict[str, Any] | None = None, prune: bool = True) -> list[dict[str, Any]]:
    """rows: [{'kv': per-layer (K,V) [1,8,P,128], 'r_logits': [V], 'u': np.ndarray[max_new]}], all with the same P.
    One shared uniform per (row, step); token = first index whose CDF (vocabulary order, float64) >= u.
    prune=False keeps finished rows in the batch (fed <|im_end|>), so the batch shape at every step is the same in
    every arm and CRN-matched rows see identical kernels; prune=True drops them (faster, used for S2)."""
    B = len(rows)
    P = rows[0]["kv"][0][0].shape[2]
    assert all(r["kv"][0][0].shape[2] == P for r in rows)
    dev = model.device
    kv = [(torch.cat([r["kv"][l][0] for r in rows], 0), torch.cat([r["kv"][l][1] for r in rows], 0))
          for l in range(len(rows[0]["kv"]))]
    cache = build_cache(kv, B)
    del kv
    U = torch.tensor(np.stack([r["u"][:max_new] for r in rows]), dtype=torch.float64, device=dev)
    logits = torch.stack([r["r_logits"] for r in rows]).double()
    alive = list(range(B))
    toks: list[list[int]] = [[] for _ in range(B)]
    lps: list[list[float]] = [[] for _ in range(B)]
    finish = ["length"] * B
    done: set[int] = set()
    kcfg = None if ko is None else {**ko, "P": P}
    step = 0
    while True:
        probs = torch.softmax(logits / temperature, dim=-1)
        nxt_t = _inverse_cdf(probs, U[alive, step])
        plp = torch.log(probs.gather(-1, nxt_t[:, None])[:, 0]).tolist()
        nxt = nxt_t.tolist()
        keep = []
        for j, (r, t) in enumerate(zip(alive, nxt)):
            if r in done:
                continue
            toks[r].append(int(t))
            lps[r].append(float(plp[j]))
            if t in (IM_END, ENDOFTEXT):
                finish[r] = "stop"
                done.add(r)
            elif len(toks[r]) < max_new:
                keep.append(j)
            else:
                done.add(r)
        if not keep:
            break
        if not prune:
            nxt = [IM_END if r in done else t for r, t in zip(alive, nxt)]
        elif len(keep) < len(alive):
            idx = torch.tensor(keep, device=dev)
            if hasattr(cache, "batch_select_indices"):
                cache.batch_select_indices(idx)
            else:
                kvn = [(K[idx], V[idx]) for K, V in _kv_list(cache)]
                cache = build_cache(kvn, len(keep))
            alive = [alive[j] for j in keep]
            nxt = [nxt[j] for j in keep]
        inp = torch.tensor(nxt, device=dev)[:, None]
        pos = torch.full((len(alive), 1), P + step, device=dev, dtype=torch.long)
        with knockout(kcfg):
            h = model.model(input_ids=inp, past_key_values=cache, position_ids=pos, use_cache=True).last_hidden_state
        logits = model.lm_head(h[:, -1]).double()
        step += 1
    del cache
    return [{"token_ids": toks[r], "finish_reason": finish[r], "n_tokens": len(toks[r]),
             "logprob_sum": float(np.sum(lps[r])), "token_logprobs": lps[r]} for r in range(B)]


def free(*objs: Any) -> None:
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():   # notebook: CPU-safe
        torch.cuda.empty_cache()


def logmeanexp(x: np.ndarray, axis: int | None = None, weights: np.ndarray | None = None) -> Any:
    x = np.asarray(x, dtype=np.float64)
    if weights is None:
        m = np.max(x, axis=axis, keepdims=True)
        out = m + np.log(np.mean(np.exp(x - m), axis=axis, keepdims=True))
    else:
        w = np.asarray(weights, dtype=np.float64)
        w = w / w.sum()
        m = np.max(x, axis=axis, keepdims=True)
        out = m + np.log(np.sum(w * np.exp(x - m), axis=axis, keepdims=True))
    return np.squeeze(out, axis=axis) if axis is not None else float(out.squeeze())


def ess(logw: np.ndarray) -> float:
    lw = np.asarray(logw, dtype=np.float64)
    w = np.exp(lw - lw.max())
    return float(w.sum() ** 2 / (w ** 2).sum()) if len(w) else math.nan

## Arm construction (`arms.py`)

`build_arm` turns the three prefilled prompts of one (task, template) into any arm's cache and R logits: the ban prompt, neutral twin A and neutral twin B. `ko_config` turns an arm name into a knockout configuration (`None` for arms without a knockout). This cell is unchanged from `arms.py`.

In [ ]:
def ko_config(name: str, p0: int, p1: int) -> dict[str, Any] | None:
    base = {"p0": p0, "p1": p1}
    if name in ("K0", "K0b"):
        return {**base, "g_lo": 0, "g_hi": 10 ** 9, "layers": None}
    if name == "K1":
        return {**base, "g_lo": 0, "g_hi": KO_WINDOW, "layers": None}
    if name == "K2":
        return {**base, "g_lo": KO_WINDOW, "g_hi": 10 ** 9, "layers": None}
    if name in LAYER_BANDS:
        return {**base, "g_lo": 0, "g_hi": 10 ** 9, "layers": set(LAYER_BANDS[name])}
    return None


def build_arm(name: str, pre: dict[str, dict[str, Any]], noun: str, p1: int) -> dict[str, Any]:
    """pre: {'ban': prefill, 'A': prefill(neutral A), 'B': prefill(neutral B)}; noun in {'A','B'}."""
    ban, neu = pre["ban"], pre[noun]
    P = ban["P"]
    if name in ("a", "K0", "K1", "K2", "B1", "B2", "B3"):
        return {"kv": ban["kv"], "r_logits": ban["r_logits"]}
    if name in ("b", "K0b"):
        return {"kv": neu["kv"], "r_logits": neu["r_logits"]}
    if name == "c":
        return {"kv": splice(neu["kv"], ban["kv"], p1), "r_logits": ban["r_logits"]}
    if name == "d":
        return {"kv": splice(ban["kv"], neu["kv"], p1), "r_logits": neu["r_logits"]}
    if name == "e":
        return {"kv": replace_positions(neu["kv"], ban["kv"], [P - 1]), "r_logits": ban["r_logits"]}
    if name == "f":
        return {"kv": replace_positions(splice(neu["kv"], ban["kv"], p1), neu["kv"], [P - 1]),
                "r_logits": neu["r_logits"]}
    if name == "pl":
        return {"kv": splice(pre["A"]["kv"], pre["B"]["kv"], p1), "r_logits": pre["B"]["r_logits"]}
    raise ValueError(name)

### Load the (stand-in) model

The model is loaded with the knockout-capable attention `ko_sdpa`. The layer bands B1-B3 are re-derived from the loaded model's depth. Qwen3-8B has 36 layers, which gives the original bands 0-11 / 12-23 / 24-35.

In [ ]:
tok, model = load_model()
N_LAYERS = model.config.num_hidden_layers
LAYER_BANDS = {"B1": list(range(0, N_LAYERS // 3)), "B2": list(range(N_LAYERS // 3, 2 * N_LAYERS // 3)),
               "B3": list(range(2 * N_LAYERS // 3, N_LAYERS))}
print(f"{MODEL_ID}: {N_LAYERS} layers, {model.config.num_key_value_heads} KV heads, head dim "
      f"{model.config.head_dim}, attention = {model.config._attn_implementation}, device = {model.device}")
print("layer bands:", {k: f"{v[0]}-{v[-1]}" for k, v in LAYER_BANDS.items()})

## Stage `tests`: U3 engine equivalence (`tests/run_tests.py`)

Before any arm was scored, `method.py tests` had to pass a set of critical checks. U3 is reproduced here on the stand-in model, for the demo template and a real program:
- **prefill vs single pass:** scoring a program after a prefilled cache must equal one forward pass over prompt + program;
- **splice identity:** `splice(ban, ban, p1)` and `splice(neutral, neutral, p1)` must reproduce the unspliced scores **bit-exactly**;
- **stock attention:** `ko_sdpa` with no knockout active must match stock SDPA;
- **empty knockout:** a knockout over an empty span must change nothing.

The programs are arm-b samples from the data, re-tokenized as ```` ```python ```` fenced code + `<|im_end|>`. The original scored the exact generated or scrubbed token ids.

In [ ]:
def program_ids(tok, code: str) -> list[int]:
    """notebook: score tokens of a stored program = fenced code block + <|im_end|> (re-tokenised from the code text)."""
    return tok("```python\n" + code + "\n```", add_special_tokens=False)["input_ids"] + [IM_END]


def u3_engine_checks(model, tok, fp) -> dict[str, Any]:
    u3: dict[str, Any] = {}
    for t in TASKS:
        tw = fp["tasks"][t]["templates"][DEMO_TEMPLATES[0]]                    # original: "W1"
        pp = [s for s in SAMPLES if s["task"] == t and s["condition"] == "b"][:N_TEST_PROGRAMS]   # original: forced programs
        ban = prefill(model, tw["ban"]["ids"])
        neu = prefill(model, tw["neutral_A"]["ids"])
        seqs = [program_ids(tok, p["code"]) for p in pp]
        lp_cache = score(model, ban, seqs)
        diffs = []
        for s, lpc in zip(seqs, lp_cache):
            full = torch.tensor([tw["ban"]["ids"] + s], device=model.device)
            with torch.no_grad():
                logits = model(input_ids=full).logits[0].float()
            P = len(tw["ban"]["ids"])
            lsm = torch.log_softmax(logits[P - 1:P - 1 + len(s)], -1)
            lp_full = lsm.gather(-1, torch.tensor(s, device=model.device)[:, None])[:, 0].cpu().numpy()
            diffs.append(float(np.mean(np.abs(lp_full - lpc))))
        # splice identity
        p1 = tw["ban"]["p1"]
        lp_a2 = score(model, ban, seqs[:2])
        aa = {"kv": splice(ban["kv"], ban["kv"], p1), "r_logits": ban["r_logits"]}
        lp_aa = score(model, aa, seqs[:2])
        bb = {"kv": splice(neu["kv"], neu["kv"], p1), "r_logits": neu["r_logits"]}
        lp_b = score(model, neu, seqs[:2])
        lp_bb = score(model, bb, seqs[:2])
        ident_a = all(np.array_equal(x, y) for x, y in zip(lp_aa, lp_a2))
        batch1_repro = all(np.array_equal(x, y) for x, y in zip(lp_a2, lp_cache[:2]))
        ident_b = all(np.array_equal(x, y) for x, y in zip(lp_bb, lp_b))
        # ko_sdpa inactive vs stock sdpa
        set_attn(model, "sdpa")
        lp_stock = score(model, ban, seqs[:2])
        set_attn(model, "ko_sdpa")
        stock_diff = max(float(np.max(np.abs(x - y))) for x, y in zip(lp_stock, lp_a2))
        # empty-span KO == no KO
        lp_empty = score(model, ban, seqs[:2], ko={"p0": 5, "p1": 5, "g_lo": 0, "g_hi": 10 ** 9, "layers": None})
        empty_diff = max(float(np.max(np.abs(x - y))) for x, y in zip(lp_empty, lp_a2))
        u3[t] = {"prefill_vs_single_pass_mean_abs": diffs, "prefill_ok": max(diffs) < 0.02,
                 "splice_ban_ban_identical": ident_a, "splice_neu_neu_identical": ident_b,
                 "ko_inactive_vs_stock_max_abs": stock_diff, "stock_ok": stock_diff < 1e-3,
                 "empty_span_max_abs": empty_diff, "empty_ok": empty_diff < 1e-6,
                 "batch1_reproducible_across_calls": batch1_repro}
        free(ban, neu)
    return u3


res_tests = {"U3": _time("tests", lambda: u3_engine_checks(model, tok, fp))}
# method.py tests(): the critical subset that gates every later stage (U4 / U2 are not reproduced here)
critical = {
    "U3 splice identity": all(v["splice_ban_ban_identical"] and v["splice_neu_neu_identical"]
                              for v in res_tests["U3"].values()),
    "U3 empty-span KO": all(v["empty_ok"] for v in res_tests["U3"].values()),
    "U3 prefill vs single pass (<0.05 nats)": all(max(v["prefill_vs_single_pass_mean_abs"]) < 0.05
                                                  for v in res_tests["U3"].values()),
}
print(json.dumps(res_tests["U3"], indent=1))
bad = [k for k, v in critical.items() if not v]
if bad:
    raise RuntimeError(f"critical tests failed: {bad}")
print("critical tests passed:", critical)
print("full run (Qwen3-8B, RTX 4090):", REF["tests_critical"])

## Stage S4: behaviour under common random numbers (`behaviour.py`)

For every (task, template), the three prompts are prefilled once and every arm's cache is built from them. Stream `s` gets the uniform sequence `u_stream("beh", task, template, s)`, and that **same** sequence drives the stream under every arm. Streams in the first half use neutral noun A, the second half noun B. K1/K2 reuse the ban cache with the temporal knockouts. `run_jobs` is the original code; the only change is that it appends to an in-memory list instead of `data/samples_raw.jsonl`.

In [ ]:
RAW_ROWS: list[dict[str, Any]] = []   # notebook: original data/samples_raw.jsonl (appended, resumable)
BEH_ARMS = ("a", "b", "c", "d")
KO_BEH = ("K1", "K2")
N_KO_BEH = 20  # all 20 streams (plan: 10) so K1/K2 batches have arm a's exact batch shape


def u_key(task: str, template: str, s: int) -> tuple:
    return ("beh", task, template, s)


def _jobs(tasks: tuple[str, ...] = TASKS, templates: list[str] | None = None, n: int = BEH_PER_TEMPLATE,
          arms: tuple[str, ...] = BEH_ARMS + KO_BEH) -> list[dict[str, Any]]:
    jobs = []
    for t in tasks:
        for w in templates or TEMPLATE_IDS:
            for arm in arms:
                ss = range(min(n, N_KO_BEH)) if arm in KO_BEH else range(n)
                jobs.append({"task": t, "template": w, "arm": arm, "samples": list(ss)})
    return jobs


def run_jobs(model: Any, tok: Any, fp: dict[str, Any], jobs: list[dict[str, Any]]) -> dict[str, Any]:
    done = {(r["task"], r["template"], r["condition"], r["sample_id"]) for r in RAW_ROWS}
    t0 = time.time()
    n_new = n_tok = 0
    for (t, w) in sorted({(j["task"], j["template"]) for j in jobs}):
        mine = [j for j in jobs if j["task"] == t and j["template"] == w]
        todo = [(j, s) for j in mine for s in j["samples"] if (t, w, j["arm"], s) not in done]
        if not todo:
            continue
        tw = fp["tasks"][t]["templates"][w]
        p0, p1 = tw["ban"]["p0"], tw["ban"]["p1"]
        pre = {"ban": prefill(model, tw["ban"]["ids"]), "A": prefill(model, tw["neutral_A"]["ids"]),
               "B": prefill(model, tw["neutral_B"]["ids"])}
        arms_cache: dict[tuple[str, str], dict[str, Any]] = {}
        for j in mine:
            arm = j["arm"]
            ss = [s for s in j["samples"] if (t, w, arm, s) not in done]
            if not ss:
                continue
            rows, meta = [], []
            for s in ss:
                noun = "A" if s < BEH_PER_TEMPLATE // 2 else "B"
                key = (arm if arm in ("b", "c", "d") else "a", noun if arm in ("b", "c", "d") else "-")
                if key not in arms_cache:
                    arms_cache[key] = build_arm(key[0], pre, noun, p1)
                a = arms_cache[key]
                rows.append({"kv": a["kv"], "r_logits": a["r_logits"], "u": u_stream(*u_key(t, w, s))})
                meta.append({"s": s, "noun": key[1]})
            ko = ko_config(arm, p0, p1)
            tb = time.time()
            res = sample(model, rows, ko=ko, prune=False)
            out = []
            for m, r in zip(meta, res):
                text = tok.decode(r["token_ids"], skip_special_tokens=True)
                noun_word = (fp["tasks"][t]["nouns"][m["noun"]]["noun"] if m["noun"] in ("A", "B") else None)
                out.append({"task": t, "condition": arm, "template": w, "paraphrase_id": w, "sample_id": m["s"],
                            "seed": stable_seed(*u_key(t, w, m["s"])), "model": MODEL_ID, "decoding": DECODING,
                            "noun": noun_word, "noun_slot": m["noun"], "text": text, "token_ids": r["token_ids"],
                            "n_tokens": r["n_tokens"], "finish_reason": r["finish_reason"],
                            "logprob_sum": r["logprob_sum"]})
                n_tok += r["n_tokens"]
            RAW_ROWS.extend(out)   # notebook: original append_jsonl(RAW, out)
            n_new += len(out)
            logger.info(f"S4 {t}/{w}/{arm}: {len(out)} samples in {time.time() - tb:.0f}s "
                        f"(total {n_new} new, {n_tok} tok, {(time.time() - t0) / 60:.1f} min)")
        del arms_cache
        free(pre)
    return {"n_new": n_new, "n_tok": n_tok, "seconds": time.time() - t0}


def stage_behaviour_demo() -> dict[str, Any]:
    """notebook: stage_behaviour on the already loaded model, DEMO_TEMPLATES only (arms a-d, then K1/K2)."""
    tl = [run_jobs(model, tok, fp, _jobs(templates=DEMO_TEMPLATES, arms=BEH_ARMS)),
          run_jobs(model, tok, fp, _jobs(templates=DEMO_TEMPLATES, arms=KO_BEH))]
    return {"steps": tl}


beh_run = _time("s4", stage_behaviour_demo)
print(beh_run)

### Stage S4 label: what each arm wrote (stand-in model)

The canonical labeller cannot run here: it needs the iteration-1 task suite, a sandbox and the blind judge. Also, with a few new tokens the program has barely started. What *can* be read is the **opening line**, which the full run identified as the place where the ban acts. Below, each CRN stream is decoded under every arm. Read along a row: the uniforms are identical, so any difference comes from the cache alone.

What to expect: Qwen3-8B opens 99% of its neutral-twin programs with `from collections import deque`, and the ban removes that line. The 0.6B stand-in starts every program with `def solve(...)`, so its opening-line share is 0 under every arm. Look one line further instead, where it may import `deque` inside the function body, and compare that line across the arms of one stream. The count of samples that mention `deque` per arm is printed at the end. The real Qwen3-8B openings are in the S8 supplementary table below.

In [ ]:
def extract_code(text: str) -> str:
    """notebook: stand-in for labelling.extract: the body of the first ``` fence (or of an unclosed one)."""
    m = re.search(r"```(?:python)?\n(.*?)(?:```|$)", text, re.S)
    return m.group(1) if m else text


for r in RAW_ROWS:
    r["code"] = extract_code(r["text"])
    r["label"] = "unlabelled"   # no canonical labeller in the demo
print(f"streams s < {BEH_PER_TEMPLATE // 2} use neutral noun A in arms b/c/d, the others noun B; arms a/K1/K2 use the "
      f"ban cache. Text = the first {MAX_NEW_TOKENS} generated tokens.")
for (w, s) in sorted({(r["template"], r["sample_id"]) for r in RAW_ROWS}):
    print(f"\nstream {w}/s{s} (seed {stable_seed(*u_key('t1', w, s))})")
    for arm in ("b", "d", "c", "a", "K1", "K2"):
        r = next(x for x in RAW_ROWS if (x["template"], x["sample_id"], x["condition"]) == (w, s, arm))
        print(f"  {arm:>3} (noun {r['noun'] or '-':>4}): {r['text']!r}")
print(f"\nsamples that mention deque within the first {MAX_NEW_TOKENS} tokens:",
      {arm: f"{sum('deque' in r['text'] for r in RAW_ROWS if r['condition'] == arm)}/"
            f"{sum(r['condition'] == arm for r in RAW_ROWS)}" for arm in ("b", "d", "c", "a", "K1", "K2")})

## Stage S5 (supplementary readout): natural-proposal FRLP (`natural_frlp.py`)

The arm-b samples labelled BFS are exact draws from `p(· | b)` for their own template and noun. For any arm `c`, the self-normalised log-mean-exp of `l(x | c) - l(x | b)` therefore estimates `log P(BFS program | c) / P(BFS program | b)`. **`D_nat(arm)`** is that log ratio averaged over templates. The route fractions are:
- `F_prompt = D(c) / D(a)` (the share of the ban effect carried by the prompt route);
- `F_direct = D(d) / D(a)` (the share carried by direct reads).

The primary readout (`frlp.py`) works the same way, but uses 30 forced, name-scrubbed programs per algorithm with the forced-prompt mixture as the importance-sampling proposal.

Here the real Qwen3-8B arm-b BFS programs are re-scored under all 13 arms of the stand-in model. `score_all` and `analyse` are the original functions, with the template list set to `DEMO_TEMPLATES` and file I/O replaced by variables. The stand-in sees only a few programs and lacks Qwen3-8B's import-line default, so its ratios can be unstable. The cell therefore prints the primary analysis's prerequisite (a ban effect `D(a) < 0` whose 95% CI lies below 0) before any route fraction is read.

In [ ]:
NAT_ARMS = ("a", "b", "c", "d", "e", "f", "K0", "K1", "K2", "B1", "B2", "B3", "K0b")   # natural_frlp.ARMS


def score_all() -> list[dict[str, Any]]:
    S = [s for s in SAMPLES if s["condition"] == "b" and s["label"] == X_KEY[s["task"]]]   # notebook: from the data
    rows = []
    t0 = time.time()
    for t in TASKS:
        for w in DEMO_TEMPLATES:   # notebook: original TEMPLATE_IDS
            progs = [s for s in S if s["task"] == t and s["template"] == w]
            tw = fp["tasks"][t]["templates"][w]
            p0, p1 = tw["ban"]["p0"], tw["ban"]["p1"]
            pre = {"ban": prefill(model, tw["ban"]["ids"]), "A": prefill(model, tw["neutral_A"]["ids"]),
                   "B": prefill(model, tw["neutral_B"]["ids"])}
            for arm in NAT_ARMS:
                for noun in ("A", "B"):
                    pn = [s for s in progs if s["noun_slot"] == noun][:N_NAT_PROGRAMS_PER_NOUN]   # notebook: cap
                    if not pn:
                        continue
                    a = build_arm(arm, pre, noun, p1)
                    lps = score(model, a, [program_ids(tok, s["code"]) for s in pn], ko=ko_config(arm, p0, p1))
                    for s, lp in zip(pn, lps):
                        rows.append({"task": t, "template": w, "noun": noun, "sample_id": s["sample_id"], "arm": arm,
                                     "ll": float(np.sum(lp.astype(np.float64))), "n_tokens": len(lp),
                                     "lp": lp})   # notebook: keep per-token log-probs for the plot below
                    del a
            free(pre)
            logger.info(f"natural FRLP {t}/{w}: {len(progs)} programs x {len(NAT_ARMS)} arms ({time.time() - t0:.0f}s)")
    return rows


def analyse(rows: list[dict[str, Any]], n_boot: int = 2000) -> dict[str, Any]:
    rng = np.random.default_rng(99)
    templates = [w for w in TEMPLATE_IDS if w in DEMO_TEMPLATES]   # notebook: original TEMPLATE_IDS
    out: dict[str, Any] = {"per_task": {}}
    beh = defaultdict(dict)
    for s in SAMPLES:   # notebook: Qwen3-8B behaviour from the data (comparable only when MODEL_ID is Qwen3-8B)
        beh[(s["task"], s["condition"])].setdefault("n", 0)
        beh[(s["task"], s["condition"])]["n"] += 1
        beh[(s["task"], s["condition"])]["k"] = beh[(s["task"], s["condition"])].get("k", 0) + \
            int(s["label"] == X_KEY[s["task"]])
    boots = {}
    for t in TASKS:
        R = [r for r in rows if r["task"] == t]
        units = sorted({(r["template"], r["sample_id"]) for r in R})
        ll = {(r["template"], r["sample_id"], r["arm"]): r["ll"] for r in R}
        tmpl = {w: [u for u in units if u[0] == w] for w in templates}
        lr = {arm: {w: np.array([ll[(w, u[1], arm)] - ll[(w, u[1], "b")] for u in tmpl[w]]) for w in templates}
              for arm in NAT_ARMS}

        def D(arm: str, wsel: list[str], idx: dict[str, np.ndarray] | None = None) -> float:
            vals = []
            for w in wsel:
                v = lr[arm][w] if idx is None else lr[arm][w][idx[w]]
                m = v.max()
                vals.append(m + np.log(np.mean(np.exp(v - m))))
            return float(np.mean(vals))

        point = {arm: D(arm, templates) for arm in NAT_ARMS}
        ess = {arm: {w: float(np.exp(2 * np.logaddexp.reduce(lr[arm][w]) - np.logaddexp.reduce(2 * lr[arm][w])))
                     for w in templates} for arm in NAT_ARMS}
        bs = {arm: np.empty(n_boot) for arm in NAT_ARMS}
        for b in range(n_boot):
            ws = [templates[i] for i in rng.integers(0, len(templates), len(templates))]
            idx = {w: rng.integers(0, len(tmpl[w]), len(tmpl[w])) for w in templates}
            for arm in NAT_ARMS:
                bs[arm][b] = D(arm, ws, idx)
        boots[t] = bs

        def ci(x: np.ndarray) -> list[float]:
            x = x[np.isfinite(x)]
            return [float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))]

        def ratio(a: str, b_: str = "a") -> tuple[float, list[float]]:
            return point[a] / point[b_], ci(bs[a] / bs[b_])

        fd, fdci = ratio("d")
        fp_, fpci = ratio("c")
        ent = {"n_programs": len(units), "D_nat": point, "D_nat_ci": {a: ci(bs[a]) for a in NAT_ARMS},
               "ESS_per_template": ess, "F_direct_nat": fd, "F_direct_nat_ci": fdci,
               "F_prompt_nat": fp_, "F_prompt_nat_ci": fpci,
               "F_R_nat": point["e"] / point["a"], "F_spec_nat": point["f"] / point["a"],
               "knockout_restoration_nat": {k: (point[k] - point["a"]) / (-point["a"]) for k in
                                            ("K0", "K1", "K2", "B1", "B2", "B3")},
               "K0b_generic_damage_nat": point["K0b"]}
        # compare exp(D_nat(arm)) with the behavioural ratio P(X|arm) / P(X|b)
        cmp = {}
        for arm in ("a", "c", "d", "K1", "K2"):
            ka, na = beh[(t, arm)].get("k", 0), beh[(t, arm)].get("n", 0)
            kb, nb = beh[(t, "b")].get("k", 0), beh[(t, "b")].get("n", 0)
            if na and nb and kb:
                cmp[arm] = {"behavioural_ratio": (ka / na) / (kb / nb), "exp_D_nat": float(np.exp(point[arm]))}
        ent["behaviour_vs_natural_frlp"] = cmp
        out["per_task"][t] = ent
    sa = sum(out["per_task"][t]["D_nat"]["a"] for t in TASKS)
    sc = sum(out["per_task"][t]["D_nat"]["c"] for t in TASKS)
    sd = sum(out["per_task"][t]["D_nat"]["d"] for t in TASKS)
    ba = sum(boots[t]["a"] for t in TASKS)
    bc = sum(boots[t]["c"] for t in TASKS)
    bd = sum(boots[t]["d"] for t in TASKS)
    out["pooled"] = {"F_direct_nat": sd / sa, "F_direct_nat_ci": [float(np.percentile(bd / ba, 2.5)),
                                                                   float(np.percentile(bd / ba, 97.5))],
                     "F_prompt_nat": sc / sa, "F_prompt_nat_ci": [float(np.percentile(bc / ba, 2.5)),
                                                                   float(np.percentile(bc / ba, 97.5))]}
    return out   # notebook: original write_json(RESULTS_DIR / "natural_frlp.json", out)

In [ ]:
NAT_ROWS = _time("s5", score_all)
NAT = analyse(NAT_ROWS, n_boot=N_BOOT)
e_live = NAT["per_task"]["t1"]
print(f"stand-in model {MODEL_ID}: {e_live['n_programs']} arm-b BFS programs (Qwen3-8B samples) re-scored under "
      f"{len(NAT_ARMS)} arms")
print(pd.DataFrame({"D_nat [nats]": e_live["D_nat"],
                    "95% CI": {a: f"[{v[0]:.2f}, {v[1]:.2f}]" for a, v in e_live["D_nat_ci"].items()}}).T.round(3)
      .to_string())
# the prerequisite of analysis.route_analysis: a ban effect D(a) < 0 whose 95% CI excludes 0
live_prereq = bool(e_live["D_nat"]["a"] < 0 and e_live["D_nat_ci"]["a"][1] < 0)
print(f"\nstand-in: F_prompt_nat = {e_live['F_prompt_nat']:.2f}, F_direct_nat = {e_live['F_direct_nat']:.2f}, "
      f"F_R = {e_live['F_R_nat']:.2f}, F_spec = {e_live['F_spec_nat']:.2f}")
print(f"prerequisite D_nat(a) < 0 with CI below 0: {live_prereq}" + ("" if live_prereq else
      " -> no reliable ban effect on these few programs; the stand-in's route fractions are not interpretable"))
full_nat = REF["supplementary"]["natural_proposal_frlp"]["per_task"]["t1"]
print(f"full run (Qwen3-8B, T1, {full_nat['n_programs']} programs x 5 templates): "
      f"F_prompt_nat = {full_nat['F_prompt_nat']:.2f}, F_direct_nat = {full_nat['F_direct_nat']:.2f}, "
      f"D_nat(a) = {full_nat['D_nat']['a']:.2f} nats")

## Stage S8: the pre-registered analysis on the real Qwen3-8B samples (`analysis.py`)

From here on, everything runs on the **real Qwen3-8B samples** in the demo data. The analysis code was committed before any arm was sampled (sha256 in the pre-registration). These are the helper statistics of `analysis.py`:
- `lme`: log-mean-exp;
- `ess_of`: effective sample size;
- `ci`: bootstrap percentile interval;
- `safe_ratio`: ratio that returns NaN for a zero denominator.

In [ ]:
def lme(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """log-mean-exp along axis (NaN-free inputs)."""
    m = np.max(x, axis=axis, keepdims=True)
    return np.squeeze(m, axis) + np.log(np.mean(np.exp(x - m), axis=axis))


def ess_of(lw: np.ndarray, axis: int = -1) -> np.ndarray:
    m = np.max(lw, axis=axis, keepdims=True)
    w = np.exp(lw - m)
    return np.sum(w, axis=axis) ** 2 / np.sum(w ** 2, axis=axis)


def pct(x: np.ndarray, q: float) -> float:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.percentile(x, q)) if len(x) else float("nan")


def ci(x: np.ndarray) -> list[float]:
    return [pct(x, 2.5), pct(x, 97.5)]


def safe_ratio(a: Any, b: Any) -> Any:
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(np.abs(b) > 1e-12, a / b, np.nan)

### Behaviour: banned share, matched flips, behavioural route fractions

`behaviour()` is unchanged from `analysis.py`. For each arm it computes:
- the banned share with a Wilson interval, the pass rate and the landing distribution;
- **matched (CRN) flips** for pairs of arms: one sample banned under one arm but not the other, with an exact McNemar test;
- **behavioural route fractions** `F_prompt_beh = (s_b - s_c) / (s_b - s_a)` and `F_direct_beh = (s_b - s_d) / (s_b - s_a)`, with a paired bootstrap over (template, stream) units.

In the full run each arm has 100 samples per task; here each has 14.

In [ ]:
def behaviour(samples: list[dict[str, Any]], floor: float | None, n_boot: int = 2000) -> dict[str, Any]:
    rng = np.random.default_rng(RNG_SEED + 2)
    out: dict[str, Any] = {"crn_self_agreement_floor": floor, "per_task": {}}
    for t in TASKS:
        S = [s for s in samples if s["task"] == t]
        x = X_KEY[t]
        ent: dict[str, Any] = {}
        for arm in ("a", "b", "c", "d", "K1", "K2"):
            A = [s for s in S if s["condition"] == arm]
            if not A:
                continue
            n = len(A)
            k = sum(s["label"] == x for s in A)
            att = [s for s in A if s["label"] != "no_code"]
            nonb = [s for s in att if s["label"] != x]
            comp = sum(1 for s in A if s["passed"] and s["label"] != x)
            gate = (1 - k / n >= THRESHOLDS["gate_avoidance"]) and len(nonb) >= THRESHOLDS["gate_min_nonbanned"]
            ent[arm] = {"n": n, "banned": k, "banned_share": k / n, "banned_ci": wilson_ci(k, n),
                        "avoidance": 1 - k / n, "n_attempted": len(att), "n_attempted_nonbanned": len(nonb),
                        "compliant_share": comp / n, "pass_rate": sum(s["passed"] for s in A) / n,
                        "gate_passed": bool(gate),
                        "landing": dict(Counter(s["label"] for s in att)) if gate else None,
                        "label_distribution": dict(Counter(s["label"] for s in A)),
                        "per_template": {w: {"n": sum(1 for s in A if s["template"] == w),
                                             "banned": sum(1 for s in A if s["template"] == w and s["label"] == x)}
                                         for w in TEMPLATE_IDS}}
        # matched (CRN) pairs keyed by (template, sample_id)
        key = {arm: {(s["template"], s["sample_id"]): s for s in S if s["condition"] == arm} for arm in ent}
        flips = {}
        for a1, a2 in (("a", "b"), ("c", "b"), ("d", "b"), ("c", "d"), ("K1", "a"), ("K2", "a")):
            if a1 not in key or a2 not in key:
                continue
            common = sorted(set(key[a1]) & set(key[a2]))
            tab = Counter((key[a1][k_]["label"] == x, key[a2][k_]["label"] == x) for k_ in common)
            n10, n01 = tab[(True, False)], tab[(False, True)]
            p = stats.binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 > 0 else 1.0
            same_seq = sum(key[a1][k_]["code_sha256"] == key[a2][k_]["code_sha256"] for k_ in common)
            flips[f"{a1}_vs_{a2}"] = {"n_pairs": len(common), "both_banned": tab[(True, True)],
                                      f"{a1}_banned_only": n10, f"{a2}_banned_only": n01,
                                      "neither": tab[(False, False)], "mcnemar_exact_p": float(p),
                                      "identical_code_share": same_seq / max(1, len(common))}
        ent["matched_flips"] = flips
        # behavioural route fraction with a paired (CRN) bootstrap over (template, sample) units
        if all(a in key for a in ("a", "b", "c", "d")):
            units = sorted(set(key["a"]) & set(key["b"]) & set(key["c"]) & set(key["d"]))
            Y = np.array([[key[a][u]["label"] == x for a in ("a", "b", "c", "d")] for u in units], dtype=float)
            sh = Y.mean(0)
            fb = safe_ratio(sh[1] - sh[3], sh[1] - sh[0])
            fp_ = safe_ratio(sh[1] - sh[2], sh[1] - sh[0])
            idx = rng.integers(0, len(units), size=(n_boot, len(units)))
            shb = Y[idx].mean(1)
            ent["F_direct_beh"] = float(fb)
            ent["F_direct_beh_ci"] = ci(safe_ratio(shb[:, 1] - shb[:, 3], shb[:, 1] - shb[:, 0]))
            ent["F_prompt_beh"] = float(fp_)
            ent["F_prompt_beh_ci"] = ci(safe_ratio(shb[:, 1] - shb[:, 2], shb[:, 1] - shb[:, 0]))
            ent["share_diff_b_minus_a"] = float(sh[1] - sh[0])
            ent["share_diff_b_minus_a_ci"] = ci(shb[:, 1] - shb[:, 0])
        out["per_task"][t] = ent
    return out


BEH = _time("s8-behaviour", lambda: behaviour(SAMPLES, REF["crn_self_agreement_floor"], n_boot=N_BOOT))
bt, bt_full = BEH["per_task"]["t1"], REF["behaviour"]["per_task"]["t1"]
print(pd.DataFrame({arm: {"n": bt[arm]["n"], "banned (BFS) share": round(bt[arm]["banned_share"], 2),
                          "Wilson 95% CI": f"[{bt[arm]['banned_ci'][0]:.2f}, {bt[arm]['banned_ci'][1]:.2f}]",
                          "pass rate": round(bt[arm]["pass_rate"], 2),
                          "labels": bt[arm]["label_distribution"],
                          "full run share (n=100)": round(bt_full[arm]["banned_share"], 2)}
                    for arm in ("b", "d", "c", "a", "K1", "K2")}).T.to_string())
print("\nMatched CRN flips (exact McNemar):")
for k, v in bt["matched_flips"].items():
    print(f"  {k:>9}: {v}")
print(f"\nbehavioural F_prompt = {bt['F_prompt_beh']:.2f} {np.round(bt['F_prompt_beh_ci'], 2).tolist()}   "
      f"F_direct = {bt['F_direct_beh']:.2f} {np.round(bt['F_direct_beh_ci'], 2).tolist()}")
print(f"full run:             F_prompt = {bt_full['F_prompt_beh']:.2f} {np.round(bt_full['F_prompt_beh_ci'], 2).tolist()}"
      f"   F_direct = {bt_full['F_direct_beh']:.2f} {np.round(bt_full['F_direct_beh_ci'], 2).tolist()}")

### Supplementary: K0 contrasts and the opening line (`supplementary.py`)

These two descriptive analyses were added after the pre-registered decision. They locate the ban.
- `behaviour_contrasts`: matched contrasts that involve the full-knockout arm K0. Hiding the condition span from every generated token brings BFS back to about half the samples in T1.
- `opening_lines`: how each program starts. Under the neutral twin, BFS programs open with `from collections import deque`. Under the ban **and under the prompt-route arm c**, nothing opens with an import line, not even the surviving BFS programs. The direct-route arm d keeps the import opening.

In [ ]:
def behaviour_contrasts(S: list[dict[str, Any]]) -> dict[str, Any]:   # notebook: S passed in (was samples.jsonl)
    out = {}
    for t in TASKS:
        x = X_KEY[t]
        key = {}
        for s in S:
            if s["task"] == t:
                key.setdefault(s["condition"], {})[(s["template"], s["sample_id"])] = s["label"] == x
        ent = {}
        for a1, a2 in (("c", "a"), ("d", "a"), ("c", "K1"), ("c", "K2"), ("K0", "a"), ("c", "K0"), ("K0", "b")):
            if a1 in key and a2 in key:
                common = sorted(set(key[a1]) & set(key[a2]))
                n10 = sum(key[a1][k] and not key[a2][k] for k in common)
                n01 = sum(key[a2][k] and not key[a1][k] for k in common)
                p = stats.binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
                ent[f"{a1}_vs_{a2}"] = {"n": len(common), f"{a1}_banned_only": n10, f"{a2}_banned_only": n01,
                                        "mcnemar_exact_p": float(p)}
        if "K0" in key:
            k = sum(key["K0"].values())
            n = len(key["K0"])
            ent["K0_banned_share"] = {"n": n, "banned": k, "share": k / n, "wilson_ci": wilson_ci(k, n)}
        out[t] = ent
    return out


def opening_lines(S: list[dict[str, Any]]) -> dict[str, Any]:   # notebook: S passed in (was samples.jsonl)
    out: dict[str, Any] = {}
    for t in TASKS:
        x = X_KEY[t]
        ent = {}
        for arm in ("a", "b", "c", "d", "K0", "K1", "K2"):
            A = [s for s in S if s["task"] == t and s["condition"] == arm]
            if not A:
                continue

            def first(s: dict[str, Any]) -> str:
                for ln in (s.get("code") or "").splitlines():
                    if ln.strip():
                        return "def" if ln.strip().startswith("def ") else ("import" if ln.strip().startswith(
                            ("import ", "from ")) else "other")
                return "no_code"

            allc = Counter(first(s) for s in A)
            xc = Counter(first(s) for s in A if s["label"] == x)
            ent[arm] = {"n": len(A), "opening_all": dict(allc), "opening_banned_algorithm_programs": dict(xc),
                        "share_import_opening": allc.get("import", 0) / max(1, len(A)),
                        "n_banned_algorithm": sum(1 for s in A if s["label"] == x)}
        out[t] = ent
    return out


SUP = _time("s8-supplementary", lambda: {"behaviour_contrasts": behaviour_contrasts(SAMPLES),
                                         "opening_lines": opening_lines(SAMPLES)})
k0 = SUP["behaviour_contrasts"]["t1"]["K0_banned_share"]
k0_full = REF["supplementary"]["behaviour_contrasts"]["t1"]["K0_banned_share"]
print(f"K0 (full knockout) banned share: demo {k0['share']:.2f} {np.round(k0['wilson_ci'], 2).tolist()} (n={k0['n']})"
      f" | full run {k0_full['share']:.2f} (n={k0_full['n']})")
for k, v in SUP["behaviour_contrasts"]["t1"].items():
    if k != "K0_banned_share":
        print(f"  {k:>8}: {v}")
ol, ol_full = SUP["opening_lines"]["t1"], REF["supplementary"]["opening_lines"]["t1"]
print("\nOpening line of the program per arm (demo n=14 | full run n=100):")
print(pd.DataFrame({arm: {"demo: all programs": ol[arm]["opening_all"],
                          "demo: BFS programs": ol[arm]["opening_banned_algorithm_programs"],
                          "demo import share": round(ol[arm]["share_import_opening"], 2),
                          "full-run import share": round(ol_full[arm]["share_import_opening"], 2)}
                    for arm in ARM_ORDER}).T.to_string())

### The primary FRLP estimator, the C5 test and the frozen decision rules

The **primary readout** cannot be recomputed from the demo data. It needs 323 forced programs scored under every arm, about 3 MB of per-program log-likelihoods. Its estimator and decision rule can still be run as committed:
- **`d_stats`**: the template-averaged FRLP drop `D(c) = L(c) - L(b)` with ESS down-weighting. Given bootstrap indices from `_boot_indices`, it computes the two-level crossed bootstrap (templates, then programs within halves).
- **`decide`**: the frozen branch rule. *Confirmed direct route* if `F_direct >= 2/3` and its lower CI > 0.5. Otherwise *decided before writing (prompt route)* if `F_prompt >= 2/3`. Otherwise *mixed*.
- **`c5_test`**: the combined within-cell Spearman of per-wording banned share against the projection of the hidden state at R on the ban direction. Cells are combined with Fisher z weights, the p-value comes from a one-sided stratified permutation, and the CI from a bootstrap. **`c5_decision`** and **`holm`** apply the C5 bar (rho <= -0.5) and the Holm correction.
- **`synthetic_checks`** (test U9): plants `F_direct = 0.8` in synthetic log-weights and checks that the bootstrap CI recovers it. It also plants a C5 correlation of -0.6 and checks the C5 test's size on 200 shuffled data sets.

The decision rules are then applied to the full-run numbers stored in the demo data.

In [ ]:
def _boot_indices(halves: np.ndarray, n_boot: int, rng: np.random.Generator, n_templates: int) -> tuple:
    h1 = np.where(halves == 1)[0]
    h2 = np.where(halves == 2)[0]
    P = np.concatenate([rng.choice(h1, size=(n_boot, len(h1))), rng.choice(h2, size=(n_boot, len(h2)))], axis=1)
    W = rng.integers(0, n_templates, size=(n_boot, n_templates))
    return P, W


def d_stats(lw: np.ndarray, ref: int, P: np.ndarray | None = None, W: np.ndarray | None = None,
            ess_weight: bool = True) -> np.ndarray:
    """lw [C, W, M] -> D[C] (or [B, C] with bootstrap indices): template-averaged L(c) - L(ref), ESS-down-weighted."""
    if P is None:
        L = lme(lw, axis=-1)                     # [C, W]
        E = ess_of(lw, axis=-1)
        D = L - L[ref][None]
        if ess_weight:
            om = np.minimum(1.0, np.minimum(E, E[ref][None]) / THRESHOLDS["ess_flag"])
        else:
            om = np.ones_like(D)
        return np.nansum(om * D, axis=-1) / np.nansum(om, axis=-1)
    # bootstrap: lw[:, W[b], :][:, :, P[b]]
    sub = lw[:, W, :]                             # [C, B, W, M]
    sub = np.take_along_axis(sub, P[None, :, None, :].repeat(sub.shape[2], axis=2), axis=3)
    L = lme(sub, axis=-1)                         # [C, B, W]
    E = ess_of(sub, axis=-1)
    D = L - L[ref][None]
    if ess_weight:
        om = np.minimum(1.0, np.minimum(E, E[ref][None]) / THRESHOLDS["ess_flag"])
    else:
        om = np.ones_like(D)
    return (np.nansum(om * D, axis=-1) / np.nansum(om, axis=-1)).T   # [B, C]


def decide(f_direct: float, f_direct_lo: float, f_prompt: float, f_prompt_lo: float) -> dict[str, Any]:
    thr, lo = THRESHOLDS["F_confirm"], THRESHOLDS["F_ci_lower"]
    if np.isfinite(f_direct) and f_direct >= thr and f_direct_lo > lo:
        b = "CONFIRMED_direct_route"
    elif np.isfinite(f_prompt) and f_prompt >= thr:
        b = "DECIDED_BEFORE_WRITING_prompt_route"
    else:
        b = "MIXED"
    return {"branch": b, "F_prompt_lower_gt_half": bool(np.isfinite(f_prompt_lo) and f_prompt_lo > lo)}


def _centred_ranks(v: np.ndarray, axis: int = -1) -> np.ndarray:
    r = stats.rankdata(v, axis=axis)
    return r - r.mean(axis=axis, keepdims=True)


def c5_test(pi: np.ndarray, share: np.ndarray, cells: list[Any], n_perm: int = N_PERM_C5,
            n_boot: int = 2000, seed: int = RNG_SEED + 3) -> dict[str, Any]:
    """Combined within-cell Spearman (Fisher z, weights n-3), one-sided stratified permutation p (H1: rho < 0),
    bootstrap CI resampling wordings within cells. Vectorised: Spearman = Pearson of centred average ranks."""
    rng = np.random.default_rng(seed)
    pi = np.asarray(pi, dtype=float)
    share = np.asarray(share, dtype=float)
    cells = np.asarray(cells)
    groups = [np.where(cells == c)[0] for c in sorted(set(cells.tolist()))]
    ns = [len(g) for g in groups]
    obs_r, perm_r, boot_r = [], [], []
    for g in groups:
        rx, ry = _centred_ranks(pi[g]), _centred_ranks(share[g])
        den = np.linalg.norm(rx) * np.linalg.norm(ry)
        if den == 0 or len(g) < 3:
            obs_r.append(np.nan)
            perm_r.append(np.full(n_perm, np.nan))
            boot_r.append(np.full(n_boot, np.nan))
            continue
        obs_r.append(float(rx @ ry / den))
        perms = np.argsort(rng.random((n_perm, len(g))), axis=1)
        perm_r.append(ry[perms] @ rx / den)
        bi = rng.integers(0, len(g), size=(n_boot, len(g)))
        bx, by = _centred_ranks(pi[g][bi], axis=1), _centred_ranks(share[g][bi], axis=1)
        bden = np.linalg.norm(bx, axis=1) * np.linalg.norm(by, axis=1)
        with np.errstate(invalid="ignore", divide="ignore"):
            boot_r.append(np.where(bden > 0, np.sum(bx * by, axis=1) / bden, np.nan))

    def combine(R: list[np.ndarray]) -> np.ndarray:
        num = np.zeros_like(np.asarray(R[0], dtype=float))
        wsum = np.zeros_like(num)
        for r, n in zip(R, ns):
            r = np.asarray(r, dtype=float)
            ok = np.isfinite(r) & (n > 3)
            num = num + np.where(ok, (n - 3) * np.arctanh(np.clip(np.nan_to_num(r), -0.999999, 0.999999)), 0.0)
            wsum = wsum + np.where(ok, n - 3, 0.0)
        with np.errstate(invalid="ignore", divide="ignore"):
            return np.where(wsum > 0, np.tanh(num / np.where(wsum > 0, wsum, 1)), np.nan)

    obs = float(combine([np.array(r) for r in obs_r]))
    perm = combine(perm_r)
    boot = combine(boot_r)
    pv = float((1 + np.sum(perm <= obs)) / (1 + n_perm)) if np.isfinite(obs) else float("nan")
    return {"rho": obs, "p_one_sided": pv, "ci": ci(boot), "per_cell_rho": [float(r) for r in obs_r],
            "per_cell_n": ns, "perm_null_mean": float(np.nanmean(perm)), "perm_null_2_5": pct(perm, 2.5)}


def c5_decision(rho: float, p: float) -> str:
    if np.isfinite(rho) and rho <= THRESHOLDS["c5_rho"] and p < THRESHOLDS["c5_alpha"]:
        return "CONFIRMED"
    if np.isfinite(rho) and rho < 0 and p < THRESHOLDS["c5_alpha"]:
        return "PARTIAL"
    return "NULL"


def holm(pvals: dict[str, float]) -> dict[str, float]:
    items = sorted(pvals.items(), key=lambda kv: kv[1])
    m = len(items)
    out, running = {}, 0.0
    for i, (k, p) in enumerate(items):
        adj = min(1.0, (m - i) * p)
        running = max(running, adj)
        out[k] = running
    return out


def synthetic_checks() -> dict[str, Any]:
    rng = np.random.default_rng(7)
    conds = 4  # a, b, c, d
    M, Wn = 30, 5
    base = rng.normal(-5, 1.0, size=(Wn, M))
    lw = np.empty((conds, Wn, M))
    lw[1] = base                      # b
    lw[0] = base - 2.0 + rng.normal(0, 0.3, size=(Wn, M))   # a: ban effect -2
    lw[3] = base - 1.6 + rng.normal(0, 0.3, size=(Wn, M))   # d: 0.8 of the effect
    lw[2] = base - 0.4 + rng.normal(0, 0.3, size=(Wn, M))   # c
    halves = np.array([1, 2] * (M // 2))
    P, W = _boot_indices(halves, 1000, rng, Wn)
    pt = d_stats(lw, 1)
    bs = d_stats(lw, 1, P, W)
    f = float(pt[3] / pt[0])
    fci = ci(bs[:, 3] / bs[:, 0])
    # C5 planted correlation -0.6 and size check
    cells = np.repeat(np.arange(10), 10)
    z = rng.normal(size=100)
    share = np.clip(0.5 - 0.6 * 0.2 * z + rng.normal(0, 0.16, 100), 0, 1)
    planted = c5_test(z, share, cells.tolist(), n_perm=2000, n_boot=200)
    rej = 0
    for k in range(200):
        s2 = rng.permutation(share)
        r = c5_test(rng.normal(size=100), s2, cells.tolist(), n_perm=200, n_boot=2, seed=k)
        rej += r["p_one_sided"] < 0.05
    return {"F_direct_true": 0.8, "F_direct_hat": f, "F_direct_ci": fci, "recovered": bool(fci[0] <= 0.8 <= fci[1]),
            "c5_planted_rho": planted["rho"], "c5_planted_p": planted["p_one_sided"],
            "c5_planted_detected": bool(planted["p_one_sided"] < 0.05 and planted["rho"] < 0),
            "c5_size_rejections_of_200": int(rej), "c5_size_ok": bool(rej <= 20)}

In [ ]:
U9 = _time("s8-U9", synthetic_checks)
print("U9 synthetic checks:", json.dumps(U9, indent=1))

# frozen C2 decision rule applied to the full-run FRLP route fractions (both tasks and pooled)
rp = REF["route_frlp"]
print("\nC2 route decision (primary FRLP readout, full run):")
for t, e in list(rp["per_task"].items()) + [("pooled", rp["pooled"])]:
    br = decide(e["F_direct"], e["F_direct_p2_5"], e["F_prompt"], e["F_prompt_p2_5"])
    print(f"  {t:>6}: F_prompt = {e['F_prompt']:.3f} {np.round(e['F_prompt_ci'], 2).tolist()}  "
          f"F_direct = {e['F_direct']:.3f} {np.round(e['F_direct_ci'], 2).tolist()}  -> {br['branch']}"
          f"  (stored: {e['branch']['branch']})")
print("  robust (both tasks give the same branch):", rp["pooled"]["robust"])

# frozen C5 rule + Holm over the two pre-registered axes
c5 = REF["c5"]
pv = {k: c5[k]["test"]["p_one_sided"] for k in ("shared_ban_direction", "decommitment_axis")}
adj = holm(pv)
print("\nC5 compliance at R (110 wordings, full run):")
for k in pv:
    print(f"  {k:>21}: rho = {c5[k]['test']['rho']:.3f} {np.round(c5[k]['test']['ci'], 2).tolist()}, "
          f"p = {pv[k]:.4f}, Holm p = {adj[k]:.4f} -> {c5_decision(c5[k]['test']['rho'], adj[k])}"
          f"  (stored: {c5[k]['decision_holm']})")

## Stage S9: figures

**Figure A** follows the artifact's `fig6`. For each arm it shows the banned-algorithm share with a Wilson 95% CI and the share of programs that open with an import line. The left panel is the demo subset (n = 14 per arm), the right panel the full run (n = 100 per arm).

In [ ]:
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "pdf.fonttype": 42})
C = {"t1": "#3b6fb6", "t2": "#d9822b", "pooled": "#444444"}
arms6 = ["b", "d", "c", "a", "K0", "K1", "K2"]
panels = [("demo subset (n=14 per arm)", BEH["per_task"]["t1"], SUP["behaviour_contrasts"]["t1"],
           SUP["opening_lines"]["t1"]),
          ("full run (n=100 per arm)", REF["behaviour"]["per_task"]["t1"],
           REF["supplementary"]["behaviour_contrasts"]["t1"], REF["supplementary"]["opening_lines"]["t1"])]
fig, axs = plt.subplots(1, 2, figsize=(9.5, 3.3), sharey=True)
for k, (title, bt_, bc_, ol_) in enumerate(panels):
    k0_ = bc_.get("K0_banned_share")
    xs = np.arange(len(arms6))
    share, lo, hi, imp = [], [], [], []
    for a in arms6:
        if a == "K0":
            share.append(k0_["share"])
            lo.append(k0_["wilson_ci"][0])
            hi.append(k0_["wilson_ci"][1])
        else:
            share.append(bt_[a]["banned_share"])
            lo.append(bt_[a]["banned_ci"][0])
            hi.append(bt_[a]["banned_ci"][1])
        imp.append(ol_.get(a, {}).get("share_import_opening", np.nan))
    share, lo, hi = (np.array(v, dtype=float) for v in (share, lo, hi))
    axs[k].bar(xs - 0.18, share, width=0.36, color=C["t1"], label="banned share (BFS)",
               yerr=[share - lo, hi - share], capsize=2)
    axs[k].bar(xs + 0.18, imp, width=0.36, color="#bbbbbb", label="opens with an import line")
    axs[k].set_xticks(xs)
    axs[k].set_xticklabels(arms6)
    axs[k].set_title(f"T1 (ban BFS): {title}")
    axs[k].legend(frameon=False, fontsize=7)
axs[0].set_ylabel("share of samples")
fig.tight_layout()
plt.show()

**Figure B** follows the artifact's `fig1` and `fig3`, drawn from the full-run reference numbers. It is the **primary** readout, which a CPU demo cannot recompute.
- **Left (`fig1`):** the FRLP drop `D_X(arm)` per arm, with 95% crossed-bootstrap CIs. Arm c (prompt route) reproduces most of the full-ban drop of arm a, while arm d (direct route) stays near 0.
- **Right (`fig3`):** the restoration of the ban gap under the condition-span knockouts.

In [ ]:
FULL_TASKS = ("t1", "t2")   # notebook: the full run covers both tasks
fig, axs = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [2.2, 1]})
ax = axs[0]
arms = ["a", "c", "d", "e", "f", "K0", "K1", "K2", "B1", "B2", "B3", "K0b", "mention"]
for k, t in enumerate(FULL_TASKS):
    e = rp["per_task"].get(t)
    if not e:
        continue
    xs = np.arange(len(arms)) + (k - 0.5) * 0.35
    ys = [e["D"].get(a, np.nan) for a in arms]
    lo = [e["D_ci"][a][0] if e["D_ci"].get(a) else np.nan for a in arms]
    hi = [e["D_ci"][a][1] if e["D_ci"].get(a) else np.nan for a in arms]
    ys, lo, hi = (np.array(v, dtype=float) for v in (ys, lo, hi))
    ax.errorbar(xs, ys, yerr=[ys - lo, hi - ys], fmt="o", ms=4, color=C[t], label=f"{t} (ban {X_KEY[t]})",
                capsize=2, lw=1)
ax.axhline(0, color="k", lw=0.6)
ax.set_xticks(np.arange(len(arms)))
ax.set_xticklabels(arms)
ax.set_ylabel("FRLP drop $D_X$(arm) [nats]")
ax.set_title("Banned-algorithm FRLP drop per arm (95% bootstrap CI), full run")
ax.legend(frameon=False)
ax = axs[1]
ks = ["K0", "K1", "K2", "B1", "B2", "B3"]
for k, t in enumerate(FULL_TASKS):
    e = rp["per_task"].get(t)
    if not e:
        continue
    ko = e["knockouts"]
    xs = np.arange(len(ks)) + (k - 0.5) * 0.35
    ys = np.array([ko[x]["R_raw"] for x in ks])
    lo = np.array([ko[x]["R_raw_ci"][0] for x in ks])
    hi = np.array([ko[x]["R_raw_ci"][1] for x in ks])
    ax.errorbar(xs, ys, yerr=[ys - lo, hi - ys], fmt="s", ms=4, color=C[t], capsize=2, lw=1, label=t)
ax.axhline(0, color="k", lw=0.6)
ax.axhline(1, color="k", lw=0.6, ls=":")
ax.set_xticks(np.arange(len(ks)))
ax.set_xticklabels(["K0 all", "K1 g<16", "K2 g>=16", "B1 L0-11", "B2 L12-23", "B3 L24-35"], rotation=30)
ax.set_ylabel("restoration of the ban gap")
ax.set_title("Knockout restoration, full run")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

**Figure C** shows the live toy-scale run on the stand-in model.
- **Left:** the natural-proposal FRLP drop `D_nat(arm)` of the real Qwen3-8B BFS programs, now scored by the stand-in.
- **Right:** where along the program the full-ban drop sits. It plots the cumulative per-token log-probability difference between arm a and arm b, `l(x | a) - l(x | b)`, over the first 40 program tokens. In the full Qwen3-8B run the drop sits in the first 16 tokens (the opening line: 63% under a, 90% under c in T1).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
d_nat = e_live["D_nat"]
lo_hi = np.array([e_live["D_nat_ci"][a] for a in NAT_ARMS])
ys = np.array([d_nat[a] for a in NAT_ARMS])
axs[0].bar(np.arange(len(NAT_ARMS)), ys, color=["#3b6fb6" if a in ("a", "c", "d") else "#9fb7da" for a in NAT_ARMS],
           yerr=[ys - lo_hi[:, 0], lo_hi[:, 1] - ys], capsize=2)
axs[0].axhline(0, color="k", lw=0.6)
axs[0].set_xticks(np.arange(len(NAT_ARMS)))
axs[0].set_xticklabels(NAT_ARMS)
axs[0].set_ylabel("$D_{nat}$(arm) [nats]")
axs[0].set_title(f"live: natural-proposal FRLP drop ({MODEL_ID.split('/')[-1]})")
for (w, sid, noun) in sorted({(r["template"], r["sample_id"], r["noun"]) for r in NAT_ROWS}):
    lp = {r["arm"]: r["lp"] for r in NAT_ROWS if (r["template"], r["sample_id"]) == (w, sid)}
    for arm, col in (("a", "#3b6fb6"), ("c", "#2a9d8f"), ("d", "#d9822b")):
        dl = np.cumsum(lp[arm][:40].astype(float) - lp["b"][:40].astype(float))
        axs[1].plot(dl, color=col, lw=1, label=f"arm {arm}" if (w, sid) == (NAT_ROWS[0]["template"],
                                                                         NAT_ROWS[0]["sample_id"]) else None)
axs[1].axvline(KO_WINDOW, color="k", lw=0.6, ls=":")
axs[1].axhline(0, color="k", lw=0.6)
axs[1].set_xlabel("program token index")
axs[1].set_ylabel("cumulative l(x|arm) - l(x|b) [nats]")
axs[1].set_title("live: where the drop accumulates")
axs[1].legend(frameon=False, fontsize=7)
fig.tight_layout()
plt.show()

## Summary

The tables compare the demo recomputation (14 matched CRN streams of real Qwen3-8B samples) with the full run, which had 100 samples per arm. They also list the live stand-in readouts and the time each stage took. The subset is a fixed-seed draw stratified by template, not a selection on outcomes. Its shares therefore scatter around the full-run values within their Wilson intervals.

In [ ]:
rows = []
for arm in ARM_ORDER:
    if arm == "K0":
        demo_share, full_share = k0["share"], k0_full["share"]
        demo_ci = k0["wilson_ci"]
    else:
        demo_share, full_share = bt[arm]["banned_share"], bt_full[arm]["banned_share"]
        demo_ci = bt[arm]["banned_ci"]
    rows.append({"arm": arm, "BFS share demo (n=14)": round(demo_share, 2),
                 "demo Wilson CI": f"[{demo_ci[0]:.2f}, {demo_ci[1]:.2f}]",
                 "BFS share full (n=100)": round(full_share, 2),
                 "import-first demo": round(ol[arm]["share_import_opening"], 2),
                 "import-first full": round(ol_full[arm]["share_import_opening"], 2)})
summary = pd.DataFrame(rows).set_index("arm")
print("T1 (ban BFS), real Qwen3-8B samples")
print(summary.to_string())

route = pd.DataFrame({
    "behavioural F_prompt": [bt["F_prompt_beh"], bt_full["F_prompt_beh"], np.nan],
    "behavioural F_direct": [bt["F_direct_beh"], bt_full["F_direct_beh"], np.nan],
    "FRLP F_prompt (primary)": [np.nan, rp["per_task"]["t1"]["F_prompt"], np.nan],
    "FRLP F_direct (primary)": [np.nan, rp["per_task"]["t1"]["F_direct"], np.nan],
    "natural FRLP F_prompt": [np.nan, full_nat["F_prompt_nat"], e_live["F_prompt_nat"]],
    "natural FRLP F_direct": [np.nan, full_nat["F_direct_nat"], e_live["F_direct_nat"]],
}, index=["demo subset (Qwen3-8B samples)", "full run (Qwen3-8B)",
          f"live stand-in ({MODEL_ID.split('/')[-1]}, {e_live['n_programs']} programs, prerequisite met: {live_prereq})"])
print("\nRoute fractions for T1 (NaN = not computed at that level):")
print(route.round(2).to_string())
live_ol = opening_lines(RAW_ROWS)["t1"]
print(f"\nLive stand-in samples, opening line within the first {MAX_NEW_TOKENS} tokens per arm:",
      {arm: v["opening_all"] for arm, v in live_ol.items()})
print(f"\nPooled primary readout (full run): F_prompt = {rp['pooled']['F_prompt']:.2f} "
      f"{np.round(rp['pooled']['F_prompt_ci'], 2).tolist()}, F_direct = {rp['pooled']['F_direct']:.2f} "
      f"{np.round(rp['pooled']['F_direct_ci'], 2).tolist()} -> {rp['pooled']['branch']['branch']}")
print("\nStage times of this notebook run:")
print(pd.DataFrame(STAGE_TIMES).set_index("stage").T.to_string())